# Medical Text Simplification using GPT-2

This notebook follows the workflow of the provided BART notebook, but replaces BART with **GPT-2**.

**Important difference:** GPT-2 is a causal language model rather than an encoder-decoder model.  
Therefore, the training examples are formatted as:

`Simplify the following medical text:\nComplex text\nSimplified text:`

The model is trained to generate the simplified continuation.


In [ ]:
# Install required packages (Google Colab)
!pip -q install transformers datasets evaluate bert-score textstat nltk scikit-learn


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import nltk

from datasets import Dataset as HFDataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Load the medical text pairs

In [ ]:
# Update these paths if your dataset is stored somewhere else.
file_path_original = '/content/drive/MyDrive/MEDICAL/DATASET/complex_text.txt'
file_path_simple = '/content/drive/MyDrive/MEDICAL/DATASET/simple_text.txt'

with open(file_path_original, 'r', encoding='latin-1') as f:
    original_text = f.read()

with open(file_path_simple, 'r', encoding='latin-1') as f:
    simple_text = f.read()

original_lines = [x.strip() for x in original_text.splitlines() if x.strip()]
simple_lines = [x.strip() for x in simple_text.splitlines() if x.strip()]

# Keep paired examples only.
n = min(len(original_lines), len(simple_lines))
df = pd.DataFrame({
    'Original': original_lines[:n],
    'Simple': simple_lines[:n]
})

print("Number of pairs:", len(df))
df.head()


In [ ]:
# Inspect examples
for i in range(min(5, len(df))):
    print("ORIGINAL:")
    print(df.iloc[i]['Original'])
    print("\nSIMPLE:")
    print(df.iloc[i]['Simple'])
    print("\n" + "="*100 + "\n")


## 2. Prepare GPT-2 training text

In [ ]:
# GPT-2 learns from one continuous sequence.
# The prompt makes the task explicitly a medical-text simplification task.

PROMPT = "Simplify the following medical text:\n"
TARGET_PREFIX = "\nSimplified text: "

def make_gpt2_text(row):
    return PROMPT + row["Original"] + TARGET_PREFIX + row["Simple"]

df["gpt2_text"] = df.apply(make_gpt2_text, axis=1)

print(df["gpt2_text"].iloc[0])


## 3. Train/validation split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42
)

train_dataset = HFDataset.from_pandas(
    train_df[["gpt2_text"]].reset_index(drop=True)
)
val_dataset = HFDataset.from_pandas(
    val_df[["gpt2_text"]].reset_index(drop=True)
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))


## 4. Load GPT-2 tokenizer and model

In [ ]:
MODEL_NAME = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# GPT-2 does not have a padding token by default.
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print("Vocabulary size:", len(tokenizer))


In [ ]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["gpt2_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

# Causal LM labels are the same token sequence; the model internally
# shifts labels for next-token prediction.
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")


## 5. Fine-tune GPT-2

In [ ]:
# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

OUTPUT_DIR = "/content/drive/MyDrive/MEDICAL/Model_res/gpt2_simplification"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

trainer.train()


## 6. Save the fine-tuned GPT-2 model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved to:", OUTPUT_DIR)


## 7. Generate simplified medical text

In [ ]:
def generate_simplified_text(complex_text, max_new_tokens=100):
    model.eval()

    prompt = PROMPT + complex_text + TARGET_PREFIX
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Return only the generated portion after the target prefix.
    if TARGET_PREFIX in generated:
        generated = generated.split(TARGET_PREFIX, 1)[1]

    return generated.strip()


In [ ]:
# Test one example
sample_text = df.iloc[0]["Original"]
print("ORIGINAL:")
print(sample_text)
print("\nGPT-2 SIMPLIFIED:")
print(generate_simplified_text(sample_text))


## 8. Readability and automatic evaluation

In [ ]:
from textstat import flesch_kincaid_grade
import evaluate

bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

def evaluate_pair(prediction, reference):
    fkgl = flesch_kincaid_grade(prediction)

    bleu = bleu_metric.compute(
        predictions=[prediction],
        references=[[reference]]
    )["bleu"]

    meteor = meteor_metric.compute(
        predictions=[prediction],
        references=[reference]
    )["meteor"]

    bert = bertscore_metric.compute(
        predictions=[prediction],
        references=[reference],
        lang="en"
    )

    return {
        "FKGL": fkgl,
        "BLEU": bleu,
        "METEOR": meteor,
        "BERTScore_F1": bert["f1"][0]
    }


## 9. Evaluate on the held-out validation set

In [ ]:
results = []

# Evaluate a manageable number first. Change this to len(val_df) for the full set.
N_EVAL = min(50, len(val_df))

for _, row in val_df.iloc[:N_EVAL].iterrows():
    prediction = generate_simplified_text(row["Original"])
    scores = evaluate_pair(prediction, row["Simple"])
    results.append(scores)

results_df = pd.DataFrame(results)

print("Average validation metrics:")
display(results_df.mean(numeric_only=True).to_frame("Average Score"))


## 10. Compare original, reference, and GPT-2 output

In [ ]:
sample_df = val_df.sample(min(10, len(val_df)), random_state=42)

for _, row in sample_df.iterrows():
    generated = generate_simplified_text(row["Original"])

    print("=" * 100)
    print("ORIGINAL MEDICAL TEXT:")
    print(row["Original"])
    print("\nREFERENCE SIMPLE TEXT:")
    print(row["Simple"])
    print("\nGPT-2 GENERATED TEXT:")
    print(generated)
    print()


## 11. Optional: Load the saved model later

Use this section in a new Colab session after training has finished.


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/MEDICAL/Model_res/gpt2_simplification"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Loaded GPT-2 simplification model on:", device)


### Notes

- GPT-2 is **decoder-only**, so this notebook uses prompt-based causal-language-model fine-tuning.
- The original notebook preprocesses text with lowercasing, stop-word removal, and lemmatization. For GPT-2, this version intentionally keeps the natural text because GPT-2's pretrained tokenizer/model are designed to operate on natural language.
- `MAX_LENGTH` can be increased if your medical sentences are long and GPU memory allows it.
- If GPU memory is limited, reduce `per_device_train_batch_size` to `1` and/or reduce `MAX_LENGTH`.
- The validation metrics are calculated on generated text against the provided simplified reference text.
